# 05 â€” Chunking hiÃ©rarchique parent-enfant

Montre comment les sections Markdown deviennent de petits chunks de recherche reliÃ©s Ã  de grands parents transmis au gÃ©nÃ©rateur.

> VÃ©rifiez toujours les drapeaux `RUN_*` avant l'exÃ©cution. Pour cette reconstruction propre, les traitements locaux nÃ©cessaires des Ã©tapes 02 Ã  07 sont dÃ©jÃ  activÃ©s ; les collectes rÃ©seau et Ã©valuations GPU facultatives restent dÃ©sactivÃ©es.


In [1]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('DonnÃ©es actives :', PROJECT_ROOT / 'data')


Projet : <PROJECT_ROOT>
Python : <PROJECT_ROOT>\.venv\Scripts\python.exe
DonnÃ©es actives : <PROJECT_ROOT>\data


In [2]:
import json
import numpy as np
import pandas as pd
from chunker import Chunk, HierarchicalChunker
from config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
cleaned_dir = config.project.data_path / 'cleaned'
output_path = config.project.data_path / 'chunks' / 'chunks.jsonl'
economic_output_path = config.local_documents.chunks_full_path
facebook_cleaned_dir = config.facebook_scraping.cleaned_candidate_full_path
facebook_output_path = config.facebook_scraping.chunks_candidate_full_path
combined_path = config.project.data_path / 'chunks' / '_combined_rebuild.jsonl'
RUN_CHUNKING = True
if RUN_CHUNKING:
    all_chunks = HierarchicalChunker(config).chunk_all(cleaned_dir, combined_path)
    economic_prefix = config.local_documents.public_route_prefix.rstrip('/') + '/'
    document_chunks = [chunk for chunk in all_chunks if not chunk.url.startswith(economic_prefix)]
    economic_chunks = [chunk for chunk in all_chunks if chunk.url.startswith(economic_prefix)]

    def write_chunks(path, items):
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open('w', encoding='utf-8') as handle:
            for item in items:
                handle.write(json.dumps(item.__dict__, ensure_ascii=False) + '\n')

    write_chunks(output_path, document_chunks)
    write_chunks(economic_output_path, economic_chunks)
    combined_path.unlink(missing_ok=True)
    facebook_chunks = HierarchicalChunker(config).chunk_all(
        facebook_cleaned_dir,
        facebook_output_path,
    )
    chunks = all_chunks + facebook_chunks
    print(
        'Chunks ENSEA :', len(document_chunks),
        '| Ã©conomiques :', len(economic_chunks),
        '| Facebook :', len(facebook_chunks),
    )
    assert facebook_chunks, (
        'Aucun chunk Facebook produit. ExÃ©cutez le notebook '
        '02_collecte_facebook avant le notebook 05.'
    )
else:
    chunks = [
        Chunk(**json.loads(line))
        for path in (output_path, economic_output_path, facebook_output_path)
        for line in path.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]
print('Chunks analysÃ©s :', len(chunks))


Chunks ENSEA : 4719 | Ã©conomiques : 52486 | Facebook : 293
Chunks analysÃ©s : 57498


In [3]:
chunk_df = pd.DataFrame([chunk.__dict__ for chunk in chunks])
child_tokens_approx = chunk_df['text'].str.len() / 4
parent_tokens_approx = chunk_df['parent_text'].str.len() / 4
display(pd.Series({
    'chunks': len(chunk_df),
    'parents': chunk_df['parent_id'].nunique(),
    'documents': chunk_df['url'].nunique(),
    'enfant_p50_tokens_approx': int(child_tokens_approx.median()),
    'enfant_p95_tokens_approx': int(np.percentile(child_tokens_approx, 95)),
    'parent_p50_tokens_approx': int(parent_tokens_approx.median()),
    'parent_p95_tokens_approx': int(np.percentile(parent_tokens_approx, 95)),
}).to_frame('valeur'))
display(chunk_df.groupby(['doc_type', 'category']).size().rename('chunks').to_frame())
assert chunk_df['chunk_id'].is_unique


,valeur
chunks,57498
parents,10800
documents,1311
enfant_p50_tokens_approx,42
enfant_p95_tokens_approx,389
parent_p50_tokens_approx,111
parent_p95_tokens_approx,2098


chunks
doc_type      category                     
facebook_post facebook/AES_page          28
              facebook/CDR_page          31
              facebook/DAS_page          39
              facebook/DISE_page         56
              facebook/ECOSTAT_page      33
...                                     ...
pdf           accueil                   158
              anstat                  36874
              autres_institutions      3716
              wp-content               2159
spreadsheet   anstat                  11896

[405 rows x 1 columns]

In [4]:
sample_parent = chunk_df['parent_id'].value_counts().index[0]
sample = chunk_df[chunk_df['parent_id'].eq(sample_parent)]
display(sample[['title', 'section', 'chunk_id', 'parent_id', 'text']])
print('\nParent transmis au LLM :\n', sample.iloc[0]['parent_text'][:1500])


,title,section,chunk_id,parent_id,text
28673,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,ef323a7a7e98ee5d_53f688b16338304e_f0936dbc788c...,ef323a7a7e98ee5d_53f688b16338304e,Fait statistique exact | Indicateur : INDUSTRI...
28674,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,ef323a7a7e98ee5d_53f688b16338304e_342f5a555a91...,ef323a7a7e98ee5d_53f688b16338304e,Fait statistique exact | Indicateur : INDUSTRI...
28675,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,ef323a7a7e98ee5d_53f688b16338304e_2541e9caf983...,ef323a7a7e98ee5d_53f688b16338304e,Fait statistique exact | Indicateur : INDUSTRI...
28676,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,ef323a7a7e98ee5d_53f688b16338304e_d496aa037a59...,ef323a7a7e98ee5d_53f688b16338304e,Fait statistique exact | Indicateur : INDUSTRI...
28677,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,ef323a7a7e98ee5d_53f688b16338304e_75232891f4ad...,ef323a7a7e98ee5d_53f688b16338304e,Fait statistique exact | Indicateur : INDUSTRI...
...,...,...,...,...,...
28793,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,ef323a7a7e98ee5d_53f688b16338304e_8c81451f144c...,ef323a7a7e98ee5d_53f688b16338304e,Fait statistique exact | Indicateur : INDUSTRI...
28794,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,ef323a7a7e98ee5d_53f688b16338304e_314939809fb4...,ef323a7a7e98ee5d_53f688b16338304e,Fait statistique exact | Indicateur : INDUSTRI...
28795,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,ef323a7a7e98ee5d_53f688b16338304e_6937a6afd229...,ef323a7a7e98ee5d_53f688b16338304e,Fait statistique exact | Indicateur : INDUSTRI...
28796,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,INDICE DE PRIX DE PRODUCTION DE L’INDUSTRIE (I...,ef323a7a7e98ee5d_53f688b16338304e_c98745f10bcc...,ef323a7a7e98ee5d_53f688b16338304e,Fait statistique exact | Indicateur : INDUSTRI...



Parent transmis au LLM :
 Donnee tabulaire structuree.
Indicateur : INDUSTRIES EXTRACTIVES
Territoire : Cote d'Ivoire
Type de mesure : indice de prix
Unite : indice
Page : 1
Observations :
- Periode Août 2015: 101,404425 indice Feuille : Types Industries | 05-09
- Periode Avril 2015: 105,223848 indice Feuille : Types Industries | 05-09
- Periode Décembre 2015: 87,158282 indice Feuille : Types Industries | 05-09
- Periode Février 2015: 106,393282 indice Feuille : Types Industries | 05-09
- Periode Janvier 2015: 101,406928 indice Feuille : Types Industries | 05-09
- Periode Juillet 2015: 100,56783 indice Feuille : Types Industries | 05-09
- Periode Juin 2015: 107,981242 indice Feuille : Types Industries | 05-09
- Periode Mai 2015: 109,047419 indice Feuille : Types Industries | 05-09
- Periode Mars 2015: 104,174971 indice Feuille : Types Industries | 05-09
- Periode Novembre 2015: 88,670425 indice Feuille : Types Industries | 05-09
- Periode Octobre 2015: 93,274276 indice Feuille : Types

## Compromis retenu

Le chunk enfant d'environ 400 tokens amÃ©liore la prÃ©cision de la recherche. Le parent pouvant atteindre environ 1 800 tokens restitue le contexte complet au LLM. Le chevauchement de 15 % Ã©vite de perdre une information coupÃ©e Ã  une frontiÃ¨re. Les chunks ENSEA, Ã©conomiques et Facebook sont Ã©crits dans trois fichiers sÃ©parÃ©s afin de prÃ©server le routage des sources.


## Trois formes complÃ©mentaires de chunks

- `text` conserve les passages narratifs habituels ;
- `table_fact` reprÃ©sente exactement une cellule statistique validÃ©e : indicateur, pÃ©riode, valeur numÃ©rique complÃ¨te, unitÃ©, territoire, type de mesure et page ;
- `derived_fact` reprÃ©sente un calcul dÃ©terministe traÃ§able : total annuel de quatre flux trimestriels cohÃ©rents ou variation entre deux trimestres successifs. La formule et les cellules sources restent dans le parent.

Chaque cellule a un petit chunk enfant prÃ©cis et un parent regroupant la sÃ©rie complÃ¨te de son
indicateur. Une ligne entiÃ¨re est rejetÃ©e avant embedding si ses cellules se contredisent sur l'indicateur, la pÃ©riode, l'unitÃ©, le territoire ou la valeur.

In [5]:
content_type_summary = chunk_df.groupby(['category', 'content_type']).size().rename('chunks').to_frame()
display(content_type_summary)
structured_chunks = chunk_df[chunk_df['content_type'].eq('table_fact')].copy()
derived_chunks = chunk_df[chunk_df['content_type'].eq('derived_fact')].copy()
malformed_indicator_count = int((~structured_chunks['indicator'].map(
    lambda value: any(char.isalpha() for char in str(value))
)).sum()) if not structured_chunks.empty else 0
display(pd.Series({
    'chunks_narratifs': int(chunk_df['content_type'].eq('text').sum()),
    'chunks_faits_tabulaires': len(structured_chunks),
    'chunks_faits_calculÃ©s': len(derived_chunks),
    'lignes_tabulaires_parentes': structured_chunks['row_id'].nunique(),
    'pages_tabulaires': structured_chunks[['url', 'page']].drop_duplicates().shape[0],
    'indicateurs_sans_lettres': malformed_indicator_count,
}).to_frame('valeur'))

critical_indicator = (
    structured_chunks['indicator'].astype(str).str.normalize('NFKD')
    .str.encode('ascii', errors='ignore').str.decode('ascii').str.casefold()
)
critical_chunks = structured_chunks[
    critical_indicator.isin({'population homme', 'rapport de masculinite'})
    & structured_chunks['period'].astype(str).eq('2025')
]
assert malformed_indicator_count == 0, 'Des faits tabulaires mal alignÃ©s ont atteint le chunking.'
display(critical_chunks[['indicator', 'period', 'value', 'unit', 'page', 'text', 'parent_text']])
expected_normalized_values = {'17103895', '109.3'}
observed_normalized_values = {str(value).strip() for value in critical_chunks['normalized_value']}
assert expected_normalized_values <= observed_normalized_values, f'Les chunks atomiques attendus sont absents : {expected_normalized_values - observed_normalized_values}'
assert critical_chunks['page'].eq(10).all(), 'La provenance page 10 a Ã©tÃ© perdue.'

annual_agriculture = derived_chunks[
    derived_chunks['source_method'].eq('derived_annual_sum')
    & derived_chunks['indicator'].str.contains('AGRICULTURE VIVRIERE', case=False, na=False)
    & derived_chunks['period'].eq('2025')
    & derived_chunks['page'].eq(17)
]
pib_q3_q4 = derived_chunks[
    derived_chunks['source_method'].eq('derived_quarter_change')
    & derived_chunks['indicator'].str.contains(r'Variation de PIB.*T3 2025.*T4 2025', case=False, regex=True, na=False)
    & derived_chunks['page'].eq(14)
]
display(pd.concat([annual_agriculture, pib_q3_q4])[['indicator', 'period', 'value', 'unit', 'page', 'territory', 'measure_type', 'parent_text']])
assert '3357,6' in set(annual_agriculture['value']), 'Le total annuel agricole nâ€™a pas Ã©tÃ© calculÃ©.'
assert '-0,4' in set(pib_q3_q4['value']), 'La variation trimestrielle du PIB nâ€™a pas Ã©tÃ© calculÃ©e.'

,,chunks
category,content_type,
%f0%9d%90%80%f0%9d%90%a9%f0%9d%90%a9%f0%9d%90%9e%f0%9d%90%a5-a-%f0%9d%90%a9%f0%9d%90%ab%f0%9d%90%a8%f0%9d%90%a3%f0%9d%90%9e%f0%9d%90%ad%f0%9d%90%ac-%f0%9d%90%9d%f0%9d%90%9e-%f0%9d%90%ab,text,1
%f0%9d%90%82%f0%9d%90%8e%f0%9d%90%8d%f0%9d%90%82%f0%9d%90%8e%f0%9d%90%94%f0%9d%90%91%f0%9d%90%92-%f0%9d%90%83%f0%9d%90%84%f0%9d%90%8d%f0%9d%90%93%f0%9d%90%91%f0%9d%90%84%f0%9d%90%84,text,1
%f0%9d%90%82%f0%9d%90%a8%f0%9d%90%a7%f0%9d%90%9c%f0%9d%90%a8%f0%9d%90%ae%f0%9d%90%ab%f0%9d%90%ac-%f0%9d%90%9f%f0%9d%90%a2%f0%9d%90%a7%f0%9d%90%9a%f0%9d%90%a5-%f0%9d%90%80%f0%9d%90%92-%f0%9d%9f%90,text,1
%f0%9d%90%83%f0%9d%90%8e%f0%9d%90%94%f0%9d%90%81%f0%9d%90%8b%f0%9d%90%84-%f0%9d%90%82%f0%9d%90%84%f0%9d%90%91%f0%9d%90%84%f0%9d%90%8c%f0%9d%90%8e%f0%9d%90%8d%f0%9d%90%88%f0%9d%90%84-%f0%9d%90%83,text,5
%f0%9d%90%84%f0%9d%90%8d%f0%9d%90%92%f0%9d%90%84%f0%9d%90%80%f0%9d%90%81%f0%9d%90%82%f0%9d%90%84%f0%9d%90%80%f0%9d%90%8e-%f0%9d%90%82%f0%9d%90%8e%f0%9d%90%85%f0%9d%90%84%f0%9d%90%81-%f0%9d%90%91,text,2
...,...,...
visite-des-enseignants-de-michigan-state-university-a-yamoussoukro,text,1
visite-du-nouveau-directeur-de-la-dgri-au-cea-ensea-dabidjan,text,1
visite-dune-delegation-de-lins-de-la-guinee-conakry-a-lensea-dabidjan,text,3


,valeur
chunks_narratifs,8669
chunks_faits_tabulaires,48573
chunks_faits_calculÃ©s,256
lignes_tabulaires_parentes,5338
pages_tabulaires,449
indicateurs_sans_lettres,0


,indicator,period,value,unit,page,text,parent_text
5072,Population homme,2025,17 103 895,Nombre,10.0,Fait statistique exact | Indicateur : Populati...,Donnee tabulaire structuree.\nIndicateur : Pop...
5094,Rapport de masculinité,2025,"109,3",Nombre,10.0,Fait statistique exact | Indicateur : Rapport ...,Donnee tabulaire structuree.\nIndicateur : Rap...


,indicator,period,value,unit,page,territory,measure_type,parent_text
12062,Total annuel de AGRICULTURE VIVRIERE,2025,"3357,6",milliards de FCFA,17.0,Cote d'Ivoire,nominal,Fait statistique calculé et vérifiable\nIndica...
11875,Variation de PIB au coût des facteurs entre T3...,T3 2025 à T4 2025,"-0,2",%,14.0,Cote d'Ivoire,CVS | volume | prix chaines de 2015,Fait statistique calculé et vérifiable\nIndica...
11893,Variation de PIB entre T3 2025 et T4 2025,T3 2025 à T4 2025,"-0,4",%,14.0,Cote d'Ivoire,CVS | volume | prix chaines de 2015,Fait statistique calculé et vérifiable\nIndica...
